# Классификация ботов — quickstart

Ноутбук показывает, как загрузить данные, собрать пару простейших признаков и получить
валидный `submission.csv`. Это **не** решение задачи: скор такого baseline будет чуть выше
константы. Дальше — ваша работа.

Условие и описание метрики — в `README.md`.

In [2]:
import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [3]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [4]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [5]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


## Простейшие признаки

Два агрегата — сколько событий и сколько разных объявлений. Этого заведомо мало.

In [6]:
def basic_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
    })
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)

Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.values
Xtr.head()

,cookie_id,n_events,item_nunique
0,ck_54a059eb7d3ea68b,7,4
1,ck_7e4de46eeab82974,41,19
2,ck_9320229ef6304522,36,19
3,ck_30ccd25bc1714ed9,21,10
4,ck_a77c5f05948cdeef,29,16


## Валидация

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

Метрику берём из `metric.py` — это ровно тот код, которым считает проверяющая система.
Своя реализация почти наверняка разойдётся с официальной на одинаковых `score`:
их нельзя разделять, группа равных значений отмечается целиком.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from metric import precision_at_recall   # официальная реализация, ей же считает автопроверка

is_valid = train.window_start_ts.ge('2026-04-17').values
cols = ['n_events', 'item_nunique']

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
p_va = model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]

print('P@R0.7 на валидации:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('доля ботов (это уровень константы):', round(ytr[is_valid].mean(), 4))

P@R0.7 на валидации: 0.1025
доля ботов (это уровень константы): 0.082


## Новый пайплан

Основные улучшения:
- агрегации поведения во времени;
- энтропия и разнообразие объектов;
- признаки User-Agent;
- отношение действий к длительности окна;
- CatBoost с fallback на HistGradientBoosting;
- time-based validation;
- оптимизация под Precision@Recall >= 0.70.

In [8]:
import re
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False


def entropy_from_counts(values):
    values = np.asarray(values)
    if values.sum() == 0:
        return 0.0
    p = values / values.sum()
    return float(-(p * np.log2(p + 1e-12)).sum())


def build_behavior_features(ev, meta):
    ev = ev.copy()
    
    ev['event_ts'] = pd.to_datetime(ev['event_ts'], errors='coerce')

    ev['hour'] = ev['event_ts'].dt.hour.fillna(-1)
    ev['dow'] = ev['event_ts'].dt.dayofweek.fillna(-1)

    if 'user_agent' in ev.columns:
        ev['ua_len'] = ev['user_agent'].fillna('').astype(str).str.len()
    else:
        ev['ua_len'] = 0


    grouped = ev.groupby('cookie_id', observed=True)

    out = pd.DataFrame(index=meta['cookie_id'])


    agg_dict = {
        'events': ('event_name', 'size'),
        'unique_events': ('event_name', 'nunique'),
    }


    optional_features = [
        ('item_id', 'unique_items', 'nunique'),
        ('item_category', 'unique_categories', 'nunique'),
        ('seller_type', 'unique_sellers', 'nunique'),
        ('search_page', 'unique_pages', 'nunique'),
        ('ua_len', 'ua_len_mean', 'mean'),
        ('hour', 'hour_std', 'std'),
        ('pointer_x', 'pointer_x_std', 'std'),
        ('pointer_y', 'pointer_y_std', 'std'),
    ]


    for col, feature_name, func in optional_features:
        if col in ev.columns:
            agg_dict[feature_name] = (col, func)


    agg = grouped.agg(**agg_dict)


    out = out.merge(
        agg,
        left_index=True,
        right_index=True,
        how='left'
    )


    if 'event_name' in ev.columns:
        event_pivot = pd.crosstab(
            ev['cookie_id'],
            ev['event_name']
        )

        event_pivot.columns = [
            f'event_{c}_cnt'
            for c in event_pivot.columns
        ]

        out = out.merge(
            event_pivot,
            left_index=True,
            right_index=True,
            how='left'
        )


    def time_stats(g):
        ts = (
            g['event_ts']
            .dropna()
            .astype('int64')
            .values
        )

        if len(ts) < 2:
            return pd.Series({
                'time_span_sec': 0,
                'median_gap_sec': 0,
                'gap_std': 0
            })

        gaps = np.diff(ts) / 1e9

        return pd.Series({
            'time_span_sec': (ts[-1] - ts[0]) / 1e9,
            'median_gap_sec': np.median(gaps),
            'gap_std': np.std(gaps)
        })


    time_df = grouped.apply(time_stats)

    out = out.merge(
        time_df,
        left_index=True,
        right_index=True,
        how='left'
    )


    for col in [
        'item_id',
        'item_category',
        'seller_type'
    ]:
        if col in ev.columns:
            entropy = grouped[col].apply(
                lambda x: entropy_from_counts(
                    x.value_counts().values
                )
            )

            out[f'{col}_entropy'] = entropy


    meta_idx = meta.set_index('cookie_id')

    out = meta_idx.join(
        out,
        how='left'
    )


    if (
        'window_start_ts' in out.columns and
        'window_end_ts' in out.columns
    ):
        window_seconds = (
            pd.to_datetime(out['window_end_ts']) -
            pd.to_datetime(out['window_start_ts'])
        ).dt.total_seconds()

        out['window_seconds'] = (
            window_seconds
            .clip(lower=1)
        )

    else:
        out['window_seconds'] = 86400


    out['events_per_hour'] = (
        out['events']
        .fillna(0)
        /
        (out['window_seconds'] / 3600)
    )

    out['items_per_event'] = (
        out.get('unique_items', 0)
        /
        out['events'].clip(lower=1)
    )


    out = (
        out
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .reset_index()
    )


    return out



Xtr_adv = build_behavior_features(
    ev_tr,
    train
)

Xte_adv = build_behavior_features(
    ev_te,
    test
)


feature_cols = [
    c for c in Xtr_adv.columns
    if c != 'cookie_id'
]


print("Train shape:", Xtr_adv.shape)
print("Test shape:", Xte_adv.shape)
print("Features:", len(feature_cols))

Train shape: (11091, 33)
Test shape: (4909, 32)
Features: 32


In [9]:
valid_mask = train.window_start_ts.ge('2026-04-17').values

X_train = Xtr_adv.loc[~valid_mask, feature_cols]
X_valid = Xtr_adv.loc[valid_mask, feature_cols]

if CATBOOST_AVAILABLE:
    model_adv = CatBoostClassifier(
        iterations=800,
        depth=7,
        learning_rate=0.04,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        verbose=False,
        l2_leaf_reg=5,
        auto_class_weights='Balanced'
    )
    model_adv.fit(
        X_train,
        ytr[~valid_mask],
        eval_set=(X_valid, ytr[valid_mask]),
        early_stopping_rounds=80
    )
else:
    model_adv = HistGradientBoostingClassifier(
        max_iter=400,
        learning_rate=0.04,
        max_leaf_nodes=31,
        random_state=42
    )
    model_adv.fit(X_train, ytr[~valid_mask])

p_valid_adv = model_adv.predict_proba(X_valid)[:, 1]

print("ROC-AUC:", roc_auc_score(ytr[valid_mask], p_valid_adv))
print("P@R0.7:", precision_at_recall(ytr[valid_mask], p_valid_adv))


ROC-AUC: 1.0
P@R0.7: 1.0


## Сабмит

In [10]:


feature_cols = [
    c for c in Xtr_adv.columns
    if c not in [
        'cookie_id',
        'target'
    ]
]


feature_cols = [
    c for c in feature_cols
    if c in Xte_adv.columns
]


print("Number of features:", len(feature_cols))

assert 'target' not in feature_cols
assert 'cookie_id' not in feature_cols


valid_mask = train.window_start_ts.ge(
    '2026-04-17'
).values


X_train = Xtr_adv.loc[
    ~valid_mask,
    feature_cols
]

X_valid = Xtr_adv.loc[
    valid_mask,
    feature_cols
]


y_train = ytr[~valid_mask]
y_valid = ytr[valid_mask]


print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)



if CATBOOST_AVAILABLE:

    model_adv = CatBoostClassifier(
        iterations=800,
        depth=7,
        learning_rate=0.04,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        verbose=False,
        l2_leaf_reg=5,
        auto_class_weights='Balanced'
    )

    model_adv.fit(
        X_train,
        y_train,
        eval_set=(
            X_valid,
            y_valid
        ),
        early_stopping_rounds=80
    )

else:

    model_adv = HistGradientBoostingClassifier(
        max_iter=400,
        learning_rate=0.04,
        max_leaf_nodes=31,
        random_state=42
    )

    model_adv.fit(
        X_train,
        y_train
    )



p_valid_adv = model_adv.predict_proba(
    X_valid
)[:, 1]


print(
    "ROC-AUC:",
    roc_auc_score(
        y_valid,
        p_valid_adv
    )
)


print(
    "P@R0.7:",
    precision_at_recall(
        y_valid,
        p_valid_adv
    )
)

Number of features: 31
Train shape: (9140, 31)
Valid shape: (1951, 31)
ROC-AUC: 0.894674762702401
P@R0.7: 0.49122807017543857


## Куда копать дальше

Подсказок по конкретным признакам не будет — это и есть содержание задания. Несколько
вопросов, которые стоит себе задать:

* чем поток событий робота отличается от потока событий человека, если смотреть не на
  количество, а на **моменты времени**;
* что полезного лежит в строке `user_agent` и почему её нельзя брать как есть;
* насколько разнообразно то, что смотрит кука: объявления, категории, запросы, страницы выдачи;
* всё ли в порядке с самим файлом событий — порядок строк, дубликаты, пропуски;
* какие признаки бесполезны, потому что описывают технические характеристики, а не поведение.